In [1]:
# ==========================================
# INDIA SENTIMENT DASHBOARD (AUTO CITY DETECT)
# ==========================================

!pip install pandas textblob plotly ipywidgets -q

import pandas as pd
from textblob import TextBlob
import plotly.express as px
import ipywidgets as widgets
from collections import Counter
from IPython.display import display

# ------------------------------------------
# STEP 1: Upload CSV
# ------------------------------------------
from google.colab import files
uploaded = files.upload()

file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name)

print("✅ Dataset Loaded")
print(df.head())

# ------------------------------------------
# STEP 2: AUTO DETECT REGION/CITY COLUMN
# ------------------------------------------
region_col = None # Renamed from city_col
review_col = None

for col in df.columns:
    if "city" in col.lower() or "region" in col.lower(): # Check for both city and region
        region_col = col
    if "review" in col.lower():
        review_col = col

if region_col is None or review_col is None:
    raise Exception("❌ CSV must have 'review' and a 'city' or 'region' column")

df = df[[review_col, region_col]].dropna()
df.columns = ["review", "region"] # Renamed 'city' to 'region'

# ------------------------------------------
# STEP 3: CITY/REGION COORDINATES (EXPANDABLE)
# ------------------------------------------
# Using city_coords as region_coords since regions are cities
region_coords = {
    "Mumbai": (19.0760, 72.8777),
    "Delhi": (28.7041, 77.1025),
    "Hyderabad": (17.3850, 78.4867),
    "Chennai": (13.0827, 80.2707),
    "Bangalore": (12.9716, 77.5946),
    "Kolkata": (22.5726, 88.3639),
    "Pune": (18.5204, 73.8567),
    "Ahmedabad": (23.0225, 72.5714)
}

# Keep only regions we have coordinates for
df = df[df['region'].isin(region_coords.keys())]

# Map coordinates
df['lat'] = df['region'].map(lambda x: region_coords[x][0])
df['lon'] = df['region'].map(lambda x: region_coords[x][1])

# ------------------------------------------
# STEP 4: Sentiment Analysis
# ------------------------------------------
def get_sentiment(text):
    score = TextBlob(str(text)).sentiment.polarity
    if score > 0:
        return "Positive", score
    elif score < 0:
        return "Negative", score
    else:
        return "Neutral", score

df[['Sentiment', 'Score']] = df['review'].apply(lambda x: pd.Series(get_sentiment(x)))

# ------------------------------------------
# STEP 5: Aggregation
# ------------------------------------------
region_avg = df.groupby('region')['Score'].mean().reset_index() # Renamed 'city' to 'region'
city_counts = df.groupby(['region', 'Sentiment']).size().unstack().fillna(0).reset_index() # Renamed 'city' to 'region'

final_df = pd.merge(region_avg, city_counts, on="region") # Renamed 'city' to 'region'

final_df['lat'] = final_df['region'].map(lambda x: region_coords[x][0]) # Renamed 'city' to 'region'
final_df['lon'] = final_df['region'].map(lambda x: region_coords[x][1]) # Renamed 'city' to 'region'

# ------------------------------------------
# STEP 6: INDIA MAP
# ------------------------------------------
fig = px.scatter_geo(
    final_df,
    lat="lat",
    lon="lon",
    color="Score",
    size="Positive",
    hover_name="region", # Renamed 'city' to 'region'
    hover_data=["Score", "Positive", "Negative", "Neutral"],
    color_continuous_scale=[
        (0.0, "red"),
        (0.5, "yellow"),
        (1.0, "green")
    ],
    title="🇮🇳 India Sentiment Map (Auto Region/City Detection)" # Updated title
)

fig.update_geos(
    scope="asia",
    center={"lat": 20, "lon": 78},
    projection_scale=4
)

fig.show()

# ------------------------------------------
# STEP 7: DROPDOWN
# ------------------------------------------
dropdown = widgets.Dropdown(
    options=final_df['region'].unique().tolist(), # Renamed 'city' to 'region'
    description='Region/City:', # Updated description
    layout=widgets.Layout(width='50%')
)

output = widgets.Output()

def show_city(change):
    with output:
        output.clear_output()
        region = change['new'] # Renamed 'city' to 'region'
        data = final_df[final_df['region'] == region] # Renamed 'city' to 'region'

        print(f"\n📍 Region/City: {region}") # Updated output
        print(f"⭐ Avg Score: {round(data['Score'].values[0],2)}")
        print(f"😊 Positive: {int(data.get('Positive',0))}")
        print(f"😐 Neutral: {int(data.get('Neutral',0))}")
        print(f"😡 Negative: {int(data.get('Negative',0))}")

        words = " ".join(df[df['region']==region]['review']).lower().split() # Renamed 'city' to 'region'
        common_words = Counter(words).most_common(5)

        print("\n🔥 Trending Words:")
        for word, count in common_words:
            print(f"{word} ({count})")

dropdown.observe(show_city, names='value')

display(dropdown, output)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 44.6 MB/s eta 0:00:00


Saving sentiment_regions_500.csv to sentiment_regions_500.csv
✅ Dataset Loaded
                                      review     region sentiment
0  Excellent customer support from Hyderabad  Hyderabad  Positive
1      Amazing shopping experience in Mumbai     Mumbai  Positive
2       Bad purchase experience in Hyderabad  Hyderabad  Negative
3              Terrible service in Hyderabad  Hyderabad  Negative
4     Loved the product quality in Hyderabad  Hyderabad  Positive


Dropdown(description='Region/City:', layout=Layout(width='50%'), options=('Bangalore', 'Chennai', 'Delhi', 'Hy…

Output()